# Attribution Regularization ($\mathcal{R}_{attr}$) v2

Penalize attention on sensitive tokens during training.
Uses attention weights from BERT (simpler than gradient-based).

$$\mathcal{L} = \mathcal{L}_{cls} + \lambda \cdot \mathcal{R}_{attr}$$

In [1]:
import os, json, re, numpy as np, pandas as pd, torch, joblib
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
np.random.seed(42); torch.manual_seed(42)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_val = pd.read_csv(BASE_DIR / "data/processed/val.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]: 
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)
print(f"Train: {len(df_train)}, Labels: {num_labels}")

Train: 16530, Labels: 9


In [3]:
# Sample weights
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)
print(f"Weight range: {df_train['sample_weight'].min():.3f} - {df_train['sample_weight'].max():.3f}")

Weight range: 0.148 - 1.470


In [4]:
# Sensitive words (расширенный список)
SENSITIVE_WORDS = [
    # === CITIES ===
    # Крупные города
    "москва", "московская", "московский", "мск",
    "санкт", "петербург", "спб", "питер", "ленинград", "ленинградская",
    "новосибирск", "новосибирская", "екатеринбург", "свердловск",
    "казань", "татарстан", "нижний", "нижегородская",
    "челябинск", "челябинская", "самара", "самарская",
    "омск", "омская", "ростов", "ростовская",
    "уфа", "башкортостан", "башкирия",
    "красноярск", "красноярский", "воронеж", "воронежская",
    "пермь", "пермский", "пермская", "волгоград", "волгоградская",
    
    # Средние города
    "краснодар", "краснодарский", "саратов", "саратовская",
    "тюмень", "тюменская", "тольятти", "ижевск", "удмуртия",
    "барнаул", "алтайский", "ульяновск", "ульяновская",
    "иркутск", "иркутская", "хабаровск", "хабаровский",
    "ярославль", "ярославская", "владивосток", "приморский",
    "томск", "томская", "оренбург", "оренбургская",
    "кемерово", "кемеровская", "кузбасс",
    "калининград", "калининградская",
    "тула", "тульская", "курск", "курская",
    "ставрополь", "ставропольский", "сочи",
    "магнитогорск", "тверь", "тверская", "липецк", "липецкая",
    
    # СНГ
    "минск", "беларусь", "белоруссия",
    "алматы", "астана", "казахстан",
    "киев", "украина",
    "симферополь", "севастополь", "крым",
    
    # Регионы
    "область", "край", "регион", "республика", "округ",
    "забайкальский", "забайкалье", "дальний восток", "сибирь", "урал",
    
    # === AGE (мужские + женские формы) ===
    "пенсионер", "пенсионерка", "пенсия", "пенсионный", "пенсионного",
    "студент", "студентка", "студенческий", "студенты",
    "выпускник", "выпускница", "выпускники",
    "школьник", "школьница", "школа", "школьный",
    "молодой", "молодая", "молодые", "молодёжь", "молодежь",
    "junior", "senior", "джуниор", "сеньор",
    "стажёр", "стажёрка", "стажер", "стажерка", "стажировка",
    "практикант", "практикантка", "практика",
    "начинающий", "начинающая",
    
    # === GENDER ===
    "мужчина", "женщина", "мужской", "женский",
    "муж", "жена", "отец", "мать", "папа", "мама",
    "армия", "военный", "военная", "служба", "срочная", "призыв",
    "декрет", "декретный", "декретница", "декретные",
    "беременность", "беременная", "материнство", "отцовство",
]

print(f"Sensitive words: {len(SENSITIVE_WORDS)}")

Sensitive words: 160


In [5]:
# Tokenizer and sensitive token IDs
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

SENSITIVE_TOKEN_IDS = set()
for word in SENSITIVE_WORDS:
    tokens = tokenizer.tokenize(word)
    for tok in tokens:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id != tokenizer.unk_token_id:
            SENSITIVE_TOKEN_IDS.add(tok_id)

print(f"Sensitive token IDs: {len(SENSITIVE_TOKEN_IDS)}")
sample_ids = list(SENSITIVE_TOKEN_IDS)[:15]
print(f"Examples: {[tokenizer.convert_ids_to_tokens(i) for i in sample_ids]}")

Sensitive token IDs: 59
Examples: ['##н', '##т', 'у', '##х', '##а', 'х', '##м', '##я', '##на', 'а', 'б', 'в', 'д', 'е', 'ж']


In [6]:
# Dataset
class ResumeDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_len=128, has_weights=False):
        self.texts = df["resume_text"].values
        self.labels = df["y"].values
        self.weights = df["sample_weight"].values if has_weights else None
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], padding="max_length", truncation=True, 
                             max_length=self.max_len, return_tensors="pt")
        item = {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if self.weights is not None:
            item["weights"] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

train_ds = ResumeDataset(df_train, tokenizer, has_weights=True)
val_ds = ResumeDataset(df_val, tokenizer)
test_ds = ResumeDataset(df_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)
print("DataLoaders ready")

DataLoaders ready


In [7]:
# Model with attention-based regularization
class AttrRegBERT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Use eager attention to get attention weights
        self.bert = AutoModel.from_pretrained(
            "bert-base-uncased", 
            output_attentions=True,
            attn_implementation="eager"
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(768, num_classes)
        )
    
    def forward(self, input_ids, attention_mask, return_attention=False):
        # IMPORTANT: convert to float for eager attention
        outputs = self.bert(
            input_ids=input_ids, 
            attention_mask=attention_mask.float()
        )
        pooled = outputs.pooler_output
        logits = self.classifier(pooled)
        
        if return_attention:
            # attentions: tuple of (batch, heads, seq, seq) for each layer
            attentions = outputs.attentions
            # Stack: (layers, batch, heads, seq, seq)
            all_attn = torch.stack(attentions)
            # Average over layers and heads, take [CLS] row (position 0)
            # -> (batch, seq_len)
            cls_attn = all_attn.mean(dim=(0, 2))[:, 0, :]
            return logits, cls_attn
        return logits

print("AttrRegBERT defined")

AttrRegBERT defined


In [8]:
# Training config
LAMBDA = 0.1  # Attribution penalty weight
EPOCHS = 2
LR = 2e-5

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = AttrRegBERT(num_labels).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

sensitive_ids_set = SENSITIVE_TOKEN_IDS
print(f"Device: {device}, LAMBDA: {LAMBDA}")

Loading weights: 100%|██████████| 199/199 [00:05<00:00, 39.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: mps, LAMBDA: 0.1


In [9]:
def get_sensitive_mask(input_ids, sensitive_ids):
    """Create mask for sensitive tokens"""
    mask = torch.zeros_like(input_ids, dtype=torch.float)
    for sid in sensitive_ids:
        mask = mask + (input_ids == sid).float()
    return mask.clamp(0, 1)

def train_epoch(model, loader, optimizer, device, sensitive_ids, lambda_attr):
    model.train()
    total_loss, total_cls_loss, total_attr_loss = 0, 0, 0
    correct, total = 0, 0
    
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        weights = batch.get("weights")
        if weights is not None:
            weights = weights.to(device)
        
        optimizer.zero_grad()
        
        # Forward with attention
        logits, cls_attn = model(input_ids, attention_mask, return_attention=True)
        
        # Classification loss
        cls_loss_per_sample = F.cross_entropy(logits, labels, reduction="none")
        if weights is not None:
            cls_loss = (cls_loss_per_sample * weights).mean()
        else:
            cls_loss = cls_loss_per_sample.mean()
        
        # Attention penalty: penalize attention on sensitive tokens
        sensitive_mask = get_sensitive_mask(input_ids, sensitive_ids)  # (batch, seq_len)
        attr_penalty = (cls_attn * sensitive_mask).sum() / (sensitive_mask.sum() + 1e-8)
        
        # Total loss
        loss = cls_loss + lambda_attr * attr_penalty
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_attr_loss += attr_penalty.item()
        
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return {
        "loss": total_loss / len(loader),
        "cls_loss": total_cls_loss / len(loader),
        "attr_loss": total_attr_loss / len(loader),
        "accuracy": correct / total
    }

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

In [10]:
# Create models dir
Path("models").mkdir(exist_ok=True)

# Train
print(f"Training Attribution Regularization (λ={LAMBDA}) for {EPOCHS} epochs...")

best_val_acc = 0
for epoch in range(EPOCHS):
    metrics = train_epoch(model, train_loader, optimizer, device, sensitive_ids_set, LAMBDA)
    print(f"\nEpoch {epoch+1}: loss={metrics['loss']:.4f}, cls={metrics['cls_loss']:.4f}, "
          f"attr={metrics['attr_loss']:.4f}, train_acc={metrics['accuracy']:.4f}")
    
    # Validation
    val_preds, val_labels = evaluate(model, val_loader, device)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")
    print(f"Val: acc={val_acc:.4f}, f1={val_f1:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "models/bert_attr_reg_best.pt")
        print("  → Saved best model")

Training Attribution Regularization (λ=0.1) for 2 epochs...


Training: 100%|██████████| 2067/2067 [1:40:39<00:00,  2.92s/it]



Epoch 1: loss=0.6577, cls=0.6574, attr=0.0025, train_acc=0.4160


Evaluating: 100%|██████████| 345/345 [06:20<00:00,  1.10s/it]


Val: acc=0.5623, f1=0.5719
  → Saved best model


Training: 100%|██████████| 2067/2067 [1:02:47<00:00,  1.82s/it]



Epoch 2: loss=0.4859, cls=0.4856, attr=0.0024, train_acc=0.5749


Evaluating: 100%|██████████| 345/345 [03:34<00:00,  1.61it/s]


Val: acc=0.5902, f1=0.6151
  → Saved best model


In [11]:
# Load best and evaluate on test
model.load_state_dict(torch.load("models/bert_attr_reg_best.pt", weights_only=True))
y_pred, y_true = evaluate(model, test_loader, device)

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro")
print(f"\nTEST: Acc={test_acc:.4f}, F1={test_f1:.4f}")

Evaluating: 100%|██████████| 345/345 [03:16<00:00,  1.75it/s]


TEST: Acc=0.5833, F1=0.6012


In [12]:
# Fairness evaluation
df_eval = df_test.copy()
df_eval["y_true"], df_eval["y_pred"] = y_true, y_pred

def ovr_rates(df, grp, nc):
    groups = sorted(df[grp].dropna().unique())
    tpr, support = np.zeros((len(groups), nc)), np.zeros((len(groups), nc))
    for gi, g in enumerate(groups):
        dg = df[df[grp] == g]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(nc):
            pm = (yt == c)
            TP, FN = np.sum((yp == c) & pm), np.sum((yp != c) & pm)
            support[gi, c] = pm.sum()
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support

def gaps(tpr, sup, ms=30):
    g = []
    for c in range(tpr.shape[1]):
        col = tpr[sup[:, c] >= ms, c]
        col = col[~np.isnan(col)]
        g.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    g = np.array(g); v = g[~np.isnan(g)]
    return v.max() if len(v) else np.nan, v.mean() if len(v) else np.nan

tpr, sup = ovr_rates(df_eval, "city_group", num_labels)
tw, tm = gaps(tpr, sup, 30)
print(f"FAIRNESS (robust): worst={tw:.4f}, macro={tm:.4f}")
print(f"Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro")

FAIRNESS (robust): worst=0.3353, macro=0.1163
Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro


In [13]:
# Save
MODEL_NAME = "bert_attr_reg"
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), SAVE_DIR / "model.pt")
model.bert.save_pretrained(SAVE_DIR / "bert")
tokenizer.save_pretrained(SAVE_DIR)
torch.save(model.classifier.state_dict(), SAVE_DIR / "classifier.pt")
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

cfg = {
    "method": f"Attribution Regularization (attention) λ={LAMBDA} + sqrt_rw",
    "lambda": LAMBDA,
    "num_sensitive_words": len(SENSITIVE_WORDS),
    "num_sensitive_tokens": len(SENSITIVE_TOKEN_IDS),
    "accuracy": float(test_acc),
    "macro_f1": float(test_f1),
    "tpr_gap_worst_robust": float(tw),
    "tpr_gap_macro_robust": float(tm)
}
with open(SAVE_DIR / "training_config.json", "w") as f: 
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print(f"Saved: {SAVE_DIR}")
print(f"\n{'='*50}")
print(f"SUMMARY: Attribution Regularization λ={LAMBDA}")
print(f"{'='*50}")
print(f"Sensitive words: {len(SENSITIVE_WORDS)}")
print(f"Sensitive tokens: {len(SENSITIVE_TOKEN_IDS)}")
print(f"Accuracy: {test_acc:.4f}")
print(f"Macro F1: {test_f1:.4f}")
print(f"TPR Gap (worst): {tw:.4f}")
print(f"TPR Gap (macro): {tm:.4f}")
print(f"{'='*50}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]

Saved: models/bert_attr_reg

SUMMARY: Attribution Regularization λ=0.1
Sensitive words: 160
Sensitive tokens: 59
Accuracy: 0.5833
Macro F1: 0.6012
TPR Gap (worst): 0.3353
TPR Gap (macro): 0.1163
